# BIPN 162 Final Project Code Notebook
*title of project*

*Alton Gu, Margaret Jones, Laura Liang*

## idk delete if theres nothing we care to write 

* hi

## Setup
*Are there packages that need to be imported, or datasets that need to be downloaded?*

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
from torch import nn, optim

from sklearn import decomposition
from sklearn.datasets import fetch_openml

from tensorflow import keras
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.layers import BatchNormalization, Dropout, LeakyReLU
from tensorflow.keras.models import Model
from tensorflow.keras.losses import mse
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K

import tensorflow as tf

!pip install -q -U keras-tuner
from keras_tuner.tuners import Hyperband, BayesianOptimization

import IPython

import shutil
import os


In [ ]:
#checking metabolite data (Gomari paper replication)
qmdiab = pd.read_csv("QMDiab_metabolite_data.csv")
aml = pd.read_csv("AML_metabolite_data.csv")
schiz = pd.read_csv("Schizophrenia_metabolite_data.csv")

print(qmdiab.shape)
print(aml.shape)
print(schiz.shape)

print(qmdiab.columns[:10])
print(aml.columns[:10])
print(schiz.columns[:10])


## Data Wrangling

### [placeholder but new data set]

In [ ]:
df = pd.read_csv('GSE53697_RNAseq_AD.txt', delimiter='\t')
df.set_index('GeneSymbol', inplace=True)

# Isolates the RPKM, aka normalized data for analysis in a new dataframe.
rpkm_cols = [col for col in df.columns if '_rpkm' in col]
df_rpkm = df[rpkm_cols]


print(df_rpkm.head())
print(df_rpkm.shape) # There are 17 samples, and the expression levels of around 19,000 genes are measured.

In [ ]:
#Transpose the matrix so rows = genes and columns = samples
X = df_rpkm.T

print(X.shape) #the 17 samples x 19185 genes are now oriented correctly
X.head()

In [ ]:
#Remove the genes with zero variance
X = X.loc[:, X.var() > 0]

#Log transform expression since RNA-seq expression is skewed
X = np.log1p(X)

#Select highly variable genes
gene_var = X.var(axis=0)
top_genes = gene_var.sort_values(ascending=False).head(2000).index
X = X[top_genes]

#Standardize features 
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X.shape)

### Wrangling of Reference Dataset (Gomari et al.)

In [5]:
df_schizo = pd.read_csv('Schizophrenia_metabolite_data.csv')
df_schizo['Disease'] = 'Schizo'
df_AML = pd.read_csv('AML_metabolite_data.csv')
df_AML['Disease'] = 'AML'
df_diabetes = pd.read_csv('QMDiab_metabolite_data.csv')
df_diabetes['Disease'] = 'Diab'

combined_df = pd.concat([df_schizo, df_AML, df_diabetes], join='inner')
meta = combined_df[['Disease']]
combined_df_numeric = combined_df.drop(columns=['Disease'])
# combined_df_numeric = combined_df_numeric.drop(combined_df_numeric.index[0], axis=0) # Drop the first row which contains non-numeric values
combined_df_numeric = combined_df_numeric.astype(float) # Convert all values to float
print(combined_df.head())
print(combined_df.shape)

     M33228    M35186    M34214    M34419    M34395    M34389    M35628  \
0 -0.224790  0.367972  0.213836  0.034424 -0.563072 -0.822089 -0.043932   
1 -1.365763 -0.250218 -1.102545  0.106423  0.547012  0.831003 -0.235921   
2  1.361526  1.419471  0.095614  1.458757 -0.131300  0.118827  1.122446   
3 -1.129735 -0.605956 -0.677170 -0.660245  1.392298  1.474983 -0.627671   
4 -0.402553 -0.924436  0.203205  0.649445 -0.653498 -0.428191 -0.667663   

     M33230    M21127    M33955  ...    M37203    M35527    M37190    M37198  \
0  0.765081 -1.229083  0.376804  ...  0.668735 -1.156433  1.380834  1.028611   
1  0.485585 -0.849375  0.587461  ...  0.910195 -0.636947  1.442837 -0.200371   
2  0.505534 -0.656467  1.113982  ... -3.679084 -0.677344 -3.368232  1.096145   
3  0.034415 -1.640713 -0.864807  ...  0.551563 -1.768515  1.591040  1.499522   
4 -0.650306 -0.145957 -0.191565  ...  1.548937  1.241594  0.203979  2.095742   

     M38178    M39379    M38768    M37506    M37097  Disease  
0 -0.

In [6]:
from sklearn.model_selection import StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(combined_df_numeric, meta['Disease']):
    input_train, input_test = combined_df_numeric.iloc[train_index], combined_df_numeric.iloc[test_index]
    result_train, result_test = meta['Disease'].iloc[train_index], meta['Disease'].iloc[test_index]

print(f"Training set size: {len(input_train)}")
print(f"Testing set size: {len(input_test)}")

Training set size: 515
Testing set size: 129


## Data Analysis & Results

### Confirmation of VAE Pipeline (Gomari et al.)

In [9]:
# Data & model configuration
batch_size = 32
no_epochs = 1000 #lets keep it shorter for now haha
latent_dim = 18

original_dim = input_train.shape[1]
input_shape = (original_dim,)
# recommended to do this here: https://www.tensorflow.org/tutorials/keras/keras_tuner
class ClearTrainingOutput(tf.keras.callbacks.Callback):
    def on_train_end(*args, **kwargs):
     IPython.display.clear_output(wait = True)
        
        
def model_builder(hp):
    # # =================
    # # Encoder
    # # =================

    # Definition
    i       = Input(shape=input_shape, name='encoder_input')
    
    x       = Dense(hp.Int('encoder_units',
                           min_value=30,
                           max_value=180,
                           step=10))(i)
    x       = LeakyReLU()(x)
    
    mu      = Dense(latent_dim, name='latent_mu')(x)
    sigma   = Dense(latent_dim, name='latent_sigma')(x)

    # Define sampling with reparameterization trick
    def sample_z(args):
        mu, sigma = args
        batch     = K.shape(mu)[0]
        dim       = K.int_shape(mu)[1]
        eps       = K.random_normal(shape=(batch, dim))
        return mu + K.exp(sigma / 2) * eps

    # Use reparameterization trick to ....??
    z       = Lambda(sample_z, output_shape=(latent_dim, ), name='z')([mu, sigma])

    # Instantiate encoder
    encoder = Model(i, [mu, sigma, z], name='encoder')
    
    # =================
    # Decoder
    # =================

    # Definition
    d_i   = Input(shape=(latent_dim, ), name='decoder_input')
    
    x     = Dense(hp.Int('decoder_units',
                           min_value=20,
                           max_value=180,
                           step=10))(d_i)
    x     = LeakyReLU()(x)
        
    o     = Dense(original_dim)(x)

    # Instantiate decoder
    decoder = Model(d_i, o, name='decoder')
    
    # =================
    # VAE as a whole
    # =================
    beta = hp.Float('kl_beta', min_value=1e-4, max_value=1e2, sampling='LOG', default=1e-3)
    # Define loss
    def kl_reconstruction_loss(true, pred):
      # Reconstruction loss
        reconstruction_loss = mse(true, pred)
        reconstruction_loss *= original_dim

        # KL divergence loss
        kl_loss = 1 + sigma - K.square(mu) - K.exp(sigma)
        kl_loss = K.sum(kl_loss, axis=-1)
        kl_loss *= -0.5
        
        # weight KL divergence loss here
        kl_loss *= beta
        return K.mean(reconstruction_loss + kl_loss)

    # Instantiate VAE
    vae_outputs = decoder(encoder(i)[2])
    vae         = Model(i, vae_outputs, name='vae')


    # Define optimizer
    optimizer = Adam(hp.Float(
        'learning_rate',
        min_value=1e-4,
        max_value=1e-2,
        sampling='LOG',
        default=1e-3
    ))

    
    # Use a basic string for the optimizer and loss to avoid 
    # complex objects that trigger the 'experimental' flags
    vae.compile(
        optimizer='adam', 
        loss='mse',
        metrics=['mse']
    )
    
    # Manually set this to True to bypass the 'must call compile' check
    vae.compiled = True 
    
    return vae
    
# Set tuner parameters
tuner = Hyperband(
    model_builder,
    objective='mse',
    # Original is 2, upping to 3 to make tuning more aggressive and faster, since we have a lot of hyperparameters to tune and a small dataset
    factor=2,
    # Decreasing max_epochs in the interest of time
    max_epochs=20,
    directory='hyperband_optimization',
    project_name='gomari_vae',
    overwrite=True)
tuner.search_space_summary()
# Run Tuner
# Runtime on MacBook Pro 2017: 01h 15m
X_train_numeric = input_train.astype('float32').values
X_test_numeric = input_test.astype('float32').values
X_train_pure = np.array(input_train.values, dtype='float32')
X_test_pure  = np.array(input_test.values, dtype='float32')

tuner.search(
    x=X_train_pure, 
    y=X_train_pure, # Target is the INPUT
    validation_data=(X_test_pure, X_test_pure),
    epochs=20
)
# Print out best tuned parameters
tuner.results_summary(num_trials = 2)
tuner.get_best_models()[0].summary()
# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials = 1)[0]
# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials = 1)[0]

print(best_hps.get('encoder_units'))
print(best_hps.get('decoder_units'))
print(best_hps.get('learning_rate'))
print(best_hps.get('kl_beta'))


Trial 92 Complete [00h 00m 04s]
mse: 0.7230653166770935

Best mse So Far: 0.5904977321624756
Total elapsed time: 00h 04m 16s
Results summary
Results in hyperband_optimization/gomari_vae
Showing 2 best trials
Objective(name="mse", direction="min")

Trial 0038 summary
Hyperparameters:
encoder_units: 170
decoder_units: 170
kl_beta: 4.869965592729031
learning_rate: 0.002219262552270968
tuner/epochs: 20
tuner/initial_epoch: 10
tuner/bracket: 4
tuner/round: 4
tuner/trial_id: 0035
Score: 0.5904977321624756

Trial 0075 summary
Hyperparameters:
encoder_units: 170
decoder_units: 160
kl_beta: 0.02667583736042037
learning_rate: 0.0011277279217930506
tuner/epochs: 20
tuner/initial_epoch: 10
tuner/bracket: 2
tuner/round: 2
tuner/trial_id: 0072
Score: 0.5912963151931763


Model: "vae"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)      │ (None, 162)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Functional)            │ [(None, 18), (None,    │        33,866 │
│                                 │ 18), (None, 18)]       │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Functional)            │ (None, 162)            │        30,932 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 64,798 (253.12 KB)

 Trainable params: 64,798 (253.12 KB)

 Non-trainable params: 0 (0.00 B)

170
170
0.002219262552270968
4.869965592729031


In [10]:
folder_to_clean = 'hyperband_optimization'

if os.path.exists(folder_to_clean):
    shutil.rmtree(folder_to_clean)

### Analysis using PCA

In [ ]:
## YOUR CODE HERE
## FEEL FREE TO ADD MULTIPLE CELLS PER SECTION

### Analysis using VAE

In [ ]:
## code here